# Pichia-CLM Arch1 — Colab Training Notebook

**Replication of:** Narayanan & Love, PNAS 2026  
**Repo:** https://github.com/baka-44/PichiaCLM  

### Before running:
1. `Runtime → Change runtime type → GPU (T4 or A100)`
2. Run cells top to bottom
3. Model weights save to Google Drive — they persist after session ends

### Expected training time:
- T4 GPU  : ~45–60 min
- A100 GPU : ~15–20 min
- CPU only  : ~4–6 hrs (not recommended)

## Cell 1 — Mount Google Drive (persistent checkpoint storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/PichiaCLM/checkpoints'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

## Cell 2 — Clone your fork and install dependencies

In [ ]:
import os

# Clone if fresh VM, pull if repo already exists
if os.path.exists('/content/PichiaCLM/.git'):
    !git -C /content/PichiaCLM pull
else:
    !git clone https://github.com/baka-44/PichiaCLM.git /content/PichiaCLM

%cd /content/PichiaCLM
!pip install -q scikit-learn seaborn

import torch
print('PyTorch version:', torch.__version__)
print('CUDA available: ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Cell 3 — Add scripts/ to path and import modules

In [ ]:
import sys
sys.path.insert(0, '/content/PichiaCLM/scripts')

from data_prep import prepare_data, DATA_DIR, AA_MAXLEN, CDS_MAXLEN
from model import build_training_model, count_parameters, DEFAULT_HP, MAX_LENGTH, BATCH_SIZE, EPOCHS
from train import make_inputs_targets, make_dataset, make_dataloader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, os

print('All imports OK')

## Cell 4 — Prepare data
Loads all 5 CSVs, filters sequences with N, does 80/20 protein-level split,
prepends 25× augmented single-codon pairs, tokenises and pads.

In [ ]:
data = prepare_data(
    data_dir='/content/PichiaCLM/Model_PichiaCLM/Training/AllData',
    verbose=True,
)

# 80/20 split within training data for train vs validation
n_total = len(data['AA_tr'])
n_train = int(n_total * 0.80)

train_inputs, train_targets = make_dataset(data['AA_tr'][:n_train], data['Cds_tr'][:n_train], shuffle=True)
val_inputs,   val_targets   = make_dataset(data['AA_tr'][n_train:], data['Cds_tr'][n_train:], shuffle=False)
test_inputs,  test_targets  = make_inputs_targets(data['AA_ts'], data['Cds_ts'])

print(f"\nTraining sequences   : {n_train:,}")
print(f"Validation sequences : {n_total - n_train:,}")
print(f"Test sequences       : {len(data['AA_ts']):,}")
print(f"\ntrain_inputs[0].shape : {train_inputs[0].shape}  (enc_aa)")
print(f"train_inputs[1].shape : {train_inputs[1].shape}  (dec_codon)")
print(f"train_inputs[2].shape : {train_inputs[2].shape}  (dec_aa)")

## Cell 5 — Build the model

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('No GPU detected — check Runtime → Change runtime type → GPU')

print(f'GPU confirmed: {torch.cuda.get_device_name(0)}')

model = build_training_model(DEFAULT_HP, device=device)

# Print summary
total = count_parameters(model)
print(f'\nTotal parameters: {total:,}')
print(f'Expected:         9,330,664')
assert total == 9_330_664, f'Parameter count mismatch: {total}'
print('✓ Parameter count matches paper')

## Cell 6 — Set up callbacks

In [ ]:
# Callbacks are now handled inside Cell 7's training loop.
# Checkpoints, CSV logging, and early stopping are all built into train.py.
print('Ready — run Cell 7 to start training.')

## Cell 7 — Train
Early stopping monitors `val_loss` with patience=5.  
Model weights from the best epoch are restored automatically.

In [ ]:
import time
import csv
import torch
import torch.nn as nn
from train import run_epoch, make_dataloader, PAD_IDX

# ── Hyperparameters ──────────────────────────────────────────────────────────
VAL_SPLIT = 0.20
PATIENCE  = 5
SEED      = 42

torch.manual_seed(SEED)

# ── Build data loaders ───────────────────────────────────────────────────────
n_total = len(data['AA_tr'])
n_train = int(n_total * (1 - VAL_SPLIT))

tr_in,  tr_tgt  = make_dataset(
    data['AA_tr'][:n_train], data['Cds_tr'][:n_train], shuffle=True, seed=SEED)
val_in, val_tgt = make_inputs_targets(data['AA_tr'][n_train:], data['Cds_tr'][n_train:])

train_loader = make_dataloader(tr_in,  tr_tgt,  BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(val_in, val_tgt, BATCH_SIZE, shuffle=False)

print(f'Train {n_train:,}  |  Val {n_total - n_train:,}')

# ── Loss functions ───────────────────────────────────────────────────────────
crit_codon = nn.CrossEntropyLoss(ignore_index=PAD_IDX).to(device)
crit_aa    = nn.CrossEntropyLoss(ignore_index=PAD_IDX).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# ── Training loop ────────────────────────────────────────────────────────────
csv_path  = os.path.join(CHECKPOINT_DIR, 'training_history.csv')
best_path = os.path.join(CHECKPOINT_DIR, 'best_weights.pt')

best_val_loss    = float('inf')
patience_counter = 0

with open(csv_path, 'w', newline='') as f:
    csv.writer(f).writerow([
        'epoch', 'loss', 'codon_acc', 'aa_acc',
        'val_loss', 'val_codon_acc', 'val_aa_acc', 'epoch_s'])

print('\nStarting training — GPU RAM should climb after the first batch.\n')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_c_acc, tr_a_acc = run_epoch(
        model, train_loader, device, crit_codon, crit_aa, optimizer, scaler)
    val_loss, val_c_acc, val_a_acc = run_epoch(
        model, val_loader, device, crit_codon, crit_aa)

    elapsed = time.time() - t0

    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'loss={tr_loss:.4f} codon_acc={tr_c_acc:.4f}  '
          f'val_loss={val_loss:.4f} val_codon_acc={val_c_acc:.4f}  '
          f'{elapsed:.0f}s')

    # Per-epoch checkpoint to Drive
    ckpt = os.path.join(CHECKPOINT_DIR, f'pichia_clm_ep{epoch:03d}.pt')
    torch.save(model.state_dict(), ckpt)

    # CSV log
    with open(csv_path, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch, tr_loss, tr_c_acc, tr_a_acc,
            val_loss, val_c_acc, val_a_acc, elapsed])

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
        print(f'  ✓ Best model saved (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print('\nTraining complete.')
model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
print('Best weights loaded.')

## Cell 8 — Plot training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

hist_df = pd.read_csv(os.path.join(CHECKPOINT_DIR, 'training_history.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(hist_df['epoch'], hist_df['loss'],     label='Train loss')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'], label='Val loss')
axes[0].set_title('Total loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Codon accuracy
axes[1].plot(hist_df['epoch'], hist_df['codon_acc'],     label='Train codon acc')
axes[1].plot(hist_df['epoch'], hist_df['val_codon_acc'], label='Val codon acc')
axes[1].set_title('Codon prediction accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150)
plt.show()
print('Curves saved to Drive.')

## Cell 9 — Evaluate on held-out test proteins (80/20 split)

In [ ]:
print('Evaluating on held-out test proteins...')
results = model.evaluate(test_inputs, test_targets, batch_size=BATCH_SIZE, verbose=1)

print('\nTest results:')
for name, val in zip(model.metrics_names, results):
    print(f'  {name}: {val:.4f}')

## Cell 10 — Save final model weights + hyperparameters to Drive

In [ ]:
final_weights_path = os.path.join(CHECKPOINT_DIR, 'pichia_clm_arch1_final.weights.h5')
model.save_weights(final_weights_path)
print(f'Final weights saved: {final_weights_path}')

hp_path = os.path.join(CHECKPOINT_DIR, 'hyperparameters.json')
with open(hp_path, 'w') as f:
    json.dump({
        **DEFAULT_HP,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'epochs_run': len(history.history['loss']),
    }, f, indent=2)
print(f'Hyperparameters saved: {hp_path}')

print('\nAll done! Model weights are in Google Drive and will persist after session ends.')

## Cell 11 — (Optional) Resume training from a checkpoint
Run this cell instead of Cell 7 if your Colab session disconnected mid-training.

In [ ]:
# List available checkpoints
ckpts = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.h5')])
print('Available checkpoints:')
for c in ckpts:
    print(' ', c)

# Load the latest checkpoint
latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
model.load_weights(latest)
print(f'\nLoaded weights from: {latest}')
print('Now re-run Cell 7 to continue training from this epoch.')